In [3]:
#======= CREATE Project_List_API AND Project_List_API_DETAILS ===============
import os
import yaml
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "7.2-Project_List_API.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "7.2-Project_List_API_Details.csv")

# === TEST CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices','ManagedVirtualDevice','managedDevices'],
    'Unit_Test': [
        'gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
        'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
    ]
    #'Other': ['instrumentation', 'instrument']
}

# === DETECT TEST TYPES (ignoring comments) ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()
    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw in uncommented_text:
                found.add(label)
    return found

# === EXTRACT ALL API LEVELS (both matrix and hardcoded)
def extract_all_api_levels(obj):
    api_levels = set()

    def recurse(o):
        if isinstance(o, dict):
            for k, v in o.items():
                key_lower = str(k).lower()
                # Match api-level, api:, android-xyz
                if key_lower in ['api-level', 'api'] or key_lower.startswith('android-'):
                    if isinstance(v, list):
                        for val in v:
                            if str(val).isdigit():
                                api_levels.add(str(val))
                    elif isinstance(v, (int, str)) and str(v).isdigit():
                        api_levels.add(str(v))
                else:
                    recurse(v)
        elif isinstance(o, list):
            for item in o:
                recurse(item)

    recurse(obj)
    return api_levels

# === PARSE YAML FILE ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            test_types = detect_testing_types(raw)
            content = yaml.safe_load(raw)
            if not content:
                return {'types': test_types, 'api_levels': set(), 'error': True}
            all_api_levels = extract_all_api_levels(content)
            return {'types': test_types, 'api_levels': all_api_levels, 'error': False}
    except Exception:
        return {'types': set(), 'api_levels': set(), 'error': True}

# === SCAN PROJECTS ===
project_results = {}
detailed_rows = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()  # case-insensitive

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {
                    'types': set(),
                    'api_levels': set(),
                    'errors': 0,
                    'yml_count': 0
                }
                detailed_rows[project_name] = []

            project_results[project_name]['types'].update(result['types'])
            project_results[project_name]['api_levels'].update(result['api_levels'])
            project_results[project_name]['yml_count'] += 1
            if result['error']:
                project_results[project_name]['errors'] += 1

# === BUILD DETAILED ROWS ===
final_detailed_rows = []
for project, data in project_results.items(): 
    for api in data['api_levels']:
        final_detailed_rows.append({
            'project': project,
            'api_level': api,
            'source': 'detected',
            'yml_count': data['yml_count']
        })

# === EXPORT SUMMARY CSV ===
summary_rows = []
for project, result in project_results.items():
    summary_rows.append({
        'project': project,
        'test_types': ', '.join(sorted(result['types'])) if result['types'] else 'none',
        'distinct_api_levels': len(result['api_levels']),
        'yml_count': result['yml_count'],
        'yaml_errors': result['errors']
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(final_detailed_rows).to_csv(DETAILED_CSV, index=False)

print(f"\n✅ Summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Detailed CSV saved to: {DETAILED_CSV}")



✅ Summary CSV saved to: C:\GitHub\Android-Mobile-Apps\7.2-Project_List_API.csv
✅ Detailed CSV saved to: C:\GitHub\Android-Mobile-Apps\7.2-Project_List_API_Details.csv
